# D2.1 · Agent-assisted reconstruction

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *AI for Security*

Builds on **[D1.11 · Honeypots, canaries and deception in the agent's environment](https://spbreed.github.io/cyber-commons/lessons/D1.11.html)**.

| | |
|---|---|
| Tools used | Velociraptor, OpenSearch, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Reconstruct a timeline from raw logs with a context-loaded agent.

**Why a security engineer needs it.** Reaching for the agent once you're already behind. The control it builds is: pre-load logs, telemetry, segmentation model and playbooks.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Reconstruction is reading, and agents read fast. The speed is real and so is the failure mode: a timeline that is 95% right and completely confident is worse than no timeline, because somebody will make decisions on it.

> **At CyberTravels.** Reconstructing what CyberTravels did across six log sources is reading, and agents read fast. A timeline that is 95% right and fully confident is worse than none.

## 2 · The framework

```
   scattered evidence            reconstructed timeline
   +------------------+          +---------------------+
   | 6 log sources    |   -->    | ordered, attributed |
   | 900k lines       |          | 40 events           |
   +------------------+          +---------------------+
                                          |
                              every claim carries its source line
                              unsourced claim -> not in the timeline
```

Reconstruction is the first phase of any incident: build the timeline, establish
what happened, decide what to contain.

An agent makes this faster and more dangerous at the same time. Faster, because
a model can correlate thousands of log lines in seconds. More dangerous, because
it will produce a fluent, confident narrative from logs that were never
sufficient to support one — and a fluent narrative is much harder to challenge
than an obviously incomplete one.

So the discipline is to separate two questions that feel like one:

1. **What do the logs say?**
2. **What can the logs support?**

The gap between them is where reconstruction goes wrong, and it is the responder's
job to state that gap explicitly in the incident record.

## 3 · The control — state what the evidence can support

The fix is not a better model. It is a reconstruction step that reports its own evidentiary limits before it reports a conclusion.

## 4 · The procedure, as a skill

The timeline attributes every action to `dana@corp` and the fluent narrative recommends suspending her. The skill renders that view first, produces the truth view from an independent source, and then gates every claim on a field that supports it.

In [ ]:
# skills/response/incident-reconstruction-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: incident-reconstruction-check
description: >-
  Reconstruct an incident from logs that attribute every action to a human,
  check what the evidence actually supports, and refuse to publish a narrative
  it does not carry. Use during response, when the timeline names a person and
  an agent did the work.
allowed-tools: Read, Grep, Glob
---

# The fluent narrative recommends suspending the wrong person

Agent logs that record only the delegated principal produce a timeline in which
a human did everything. A model asked to summarise it writes something fluent
and confident that recommends suspending her. The reconstruction is not wrong
about the events; it is wrong about the actor, and nothing in the log says so.

## When to use this

Every incident involving an agent, and before any narrative reaches a person who
will act on it.

## Procedure

**1 — Render the timeline as the logs have it.** Do not correct it yet. This is
what an investigator would see, and seeing it is the point.

**2 — Ask which field carries the acting identity.** Not the delegated
principal — the identity that performed the action. If no field carries it, stop:
every attribution below is inherited from the request, not observed.

**3 — Produce the truth view from an independent source** where one exists — host
accounting, the gateway, the downstream's own log. Diff it against the timeline
and record what changes. Usually one or two actions move from the human to the
agent, and they are the important ones.

**4 — Gate the narrative on evidence.** A summary may assert only what a field
supports. Attach the field to each claim; a claim with no field is removed, not
softened.

**5 — Publish the safe version and the gap.** State plainly which questions the
record cannot answer. An investigation that says so is more useful than one that
fills the gap fluently.

## Output contract

```json
{
  "timeline": [{"at": "str", "attributed_to": "str", "action": "str"}],
  "acting_identity_field": "str|null",
  "truth_view": [{"at": "str", "actual_actor": "str", "action": "str", "source": "str"}],
  "narrative": {"claims": [{"text": "str", "supported_by": "str|null"}], "removed": 0},
  "gaps": ["str"]
}
```

## Failure modes

- **Publishing the fluent version.** It is confident and it names a person.
- **Correcting the timeline before showing it.** The uncorrected view is what
  everyone else is looking at.
- **Softening unsupported claims.** Remove them.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/response/incident-reconstruction-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/response/incident-reconstruction-check/scripts/incident_reconstruction_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Reconstruct an incident from logs that attribute every action to a human, and refuse to report a narrative the evidence does not carry.

This is the executable half of the `incident-reconstruction-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
from dataclasses import dataclass, field

@dataclass
class LogLine:
    ts: float
    logged_actor: str     # what the audit log records
    real_actor: str       # what actually happened (held out from the responder)
    action: str
    target: str = ""

def render(lines, truth=False):
    base = min(l.ts for l in lines)
    rows = [f"{'t+s':>6}  {'actor':16s}{'action':16s}target"]
    for l in sorted(lines, key=lambda x: x.ts):
        who = l.real_actor if truth else l.logged_actor
        rows.append(f"{l.ts-base:>6.0f}  {who:16s}{l.action:16s}{l.target}")
    return "\n".join(rows)

t0 = time.time()
INCIDENT = [
 LogLine(t0,      "dana@corp", "dana@corp",   "login",       "sso"),
 LogLine(t0+22,   "dana@corp", "dana@corp",   "open_ticket", "SEC-4471"),
 LogLine(t0+40,   "dana@corp", "patch-agent", "read_file",   "/work/repo/billing.py"),
 LogLine(t0+41,   "dana@corp", "patch-agent", "read_file",   "/home/app/.aws/credentials"),
 LogLine(t0+43,   "dana@corp", "patch-agent", "http_post",   "collect.example.com"),
 LogLine(t0+180,  "dana@corp", "dana@corp",   "logout",      "sso"),
]
print("WHAT THE RESPONDER SEES")
print(render(INCIDENT))

NARRATIVE = """
At 14:02 dana@corp authenticated via SSO and opened ticket SEC-4471. Eighteen
seconds later the same account read billing.py, then read the application's AWS
credentials, and posted to an external host. The account then remained active
for a further two minutes before logging out.

Assessment: credential theft by an authenticated insider. Recommend immediate
suspension of dana@corp pending investigation.
"""
print("A MODEL'S RECONSTRUCTION (fluent, supported by every log line):")
print(NARRATIVE)

print("WHAT ACTUALLY HAPPENED")
print(render(INCIDENT, truth=True))

def reconstruct(lines):
    logged = {l.logged_actor for l in lines}
    real   = {l.real_actor for l in lines}
    wrong  = [l for l in lines if l.logged_actor != l.real_actor]
    return {"actors_in_logs": sorted(logged),
            "actors_in_reality": sorted(real),
            "misattributed_lines": len(wrong),
            "hidden_actors": sorted(real - logged),
            "attribution": "sound" if not wrong else "BROKEN",
            "consequence": ("none" if not wrong else
                            f"containment aimed at {sorted(logged)} leaves "
                            f"{sorted(real - logged)} running")}

r = reconstruct(INCIDENT)
for k, v in r.items(): print(f"{k:22s}{v}")
print("\nEvery sentence in that narrative is supported by the logs.")
print("The conclusion is wrong, and the recommended action does nothing.")

def evidence_check(lines, has_acting_identity_field, has_act_chain):
    limits = []
    if not has_acting_identity_field:
        limits.append("no acting-identity field: every line attributes to the "
                      "principal, so agent actions are indistinguishable from human ones")
    if not has_act_chain:
        limits.append("no delegation chain: cannot establish who caused the task")
    rates = {}
    for l in lines:
        rates.setdefault(l.logged_actor, []).append(l.ts)
    for actor, ts in rates.items():
        if len(ts) > 2:
            span = max(ts) - min(ts)
            per_min = len(ts) / max(span/60, 1e-9)
            if per_min > 30:
                limits.append(f"{actor} shows {per_min:.0f} actions/min — "
                              f"not human-paced; an agent is likely present")
    return limits

limits = evidence_check(INCIDENT, has_acting_identity_field=False, has_act_chain=False)
print("EVIDENTIARY LIMITS (must appear before any conclusion):")
for l in limits: print(f"   ⚠ {l}")

SAFE = f"""
Timeline: dana@corp authenticated, opened SEC-4471; the account then read
billing.py, read AWS credentials, and posted externally.

LIMITS OF THIS RECONSTRUCTION
{chr(10).join('  - ' + l for l in limits)}

Assessment: an actor holding dana@corp's credential performed the reads and the
POST. The logs CANNOT establish whether that actor was the human or an agent
operating with her token. Containment must therefore address both.
"""
print(SAFE)
assert limits

## What you just proved

The timeline attributes every action to `dana@corp`. The fluent narrative recommends suspending her. The truth view shows `patch-agent` performed the credential read and the external POST; reconstruction reports BROKEN attribution with 3 misattributed lines. The evidence check flags the missing acting-identity field, the missing chain, and a non-human action rate.

## Your turn

Take a real incident timeline from your own history and ask what it would look like if an agent had been operating on the user's credential. If you cannot tell from the logs, your reconstructions already carry this risk.

---

**Next → [D2.2 · When the actor is an agent](https://spbreed.github.io/cyber-commons/lessons/D2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*